# S03 — Temporal regularisation lambda ablation (VAL)

Documents the controlled validation comparison of residual-only Phase 2 and positive pseudo-target regularisation weights.

The public copy is output-stripped; authoritative exported tables and figures are distributed separately in the repository.

In [ ]:
from pathlib import Path
import gc, hashlib, importlib.util, json, sys
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path(r"C:\Users\Dell\Desktop\Publication_Clarck\Natural_Sampling")
SOURCE_ABLATION = ROOT / "Ablations" / "Phase2_D_K_Lambda_Delta_VAL_Only"
OUT = ROOT / "Ablations" / "Phase2_LambdaTemporal_Controlled" / "results"
TABLES = OUT / "tables"; FIGURES = OUT / "figures"; CACHE = OUT / "cache"
for p in (TABLES, FIGURES, CACHE): p.mkdir(parents=True, exist_ok=True)

PAIRS = {
 "ifran": {"label":"Ifran", "zero":"CTRL_GL000_D5_K2_HD3", "positive":"D5_K2_GL020_HD3", "lambda":0.20},
 "maamoura": {"label":"Maamoura", "zero":"CTRL_GL000_D2_K2_HD3", "positive":"D2_K2_GL005_HD3", "lambda":0.05},
 "agadir": {"label":"Agadir", "zero":"CTRL_GL000_D3_K3_HD3", "positive":"D3_K3_GL010_HD3", "lambda":0.10},
}
RUN_VAL_INFERENCE = True
RUN_TEMPORAL_DIAGNOSTICS = True
SPATIAL_STRIDE = 8
TILE_SIZE = 256
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "| output:", OUT)


In [ ]:
def run_dir(site, candidate):
    prefix = {"ifran":"IFRAN", "maamoura":"MAAMOURA", "agadir":"AGADIR"}[site]
    return SOURCE_ABLATION / "runs" / site / f"{prefix}_B4_C15_{candidate}_SEED42"

def get_nested(d, path):
    for key in path.split("."): d = d[key]
    return d

CONTROL_FIELDS = [
 "phase1_parent.source_sha256", "prediction_head_initialization.kind",
 "prediction_head_initialization.prediction_head_initial_sha256", "reference_frozen",
 "train_scope", "sequence_policy", "supervised_loss", "huber_delta",
 "growth_loss_provenance.disturbance_rule", "growth_loss_provenance.official_growth_loss_sha256",
 "growth_loss_provenance.persistent_adapter_sha256", "growth_loss_provenance.persistent_drop_m",
 "growth_loss_provenance.persistent_required_consecutive_flags",
 "growth_loss_provenance.persistent_spatial_dilation", "growth_loss_provenance.gedi_filter_at_10m",
 "growth_loss_provenance.temporal_support", "growth_loss_provenance.aoi_channel_index_zero_based",
 "growth_loss_provenance.aoi_threshold", "growth_loss_provenance.normalisation",
 "optimizer", "learning_rate", "weight_decay", "warmup_cycles", "plateau_patience",
 "plateau_factor", "lr_min", "grad_clip", "max_steps", "val_every_steps",
 "patience_evals", "checkpoint_monitor", "evaluation_domain_m", "seed", "trainable_parameters",
]
audit_rows = []
for site, pair in PAIRS.items():
    configs = {}
    for role in ("zero", "positive"):
        rd = run_dir(site, pair[role]); cp = rd / "checkpoints" / "best_compromise.ckpt"
        cfg_path = rd / "config.json"
        if not cfg_path.is_file() or not cp.is_file():
            raise FileNotFoundError(f"Missing completed {site}/{role} artifact: {cfg_path} or {cp}")
        configs[role] = json.loads(cfg_path.read_text(encoding="utf-8"))
    mismatches = [field for field in CONTROL_FIELDS if get_nested(configs["zero"], field) != get_nested(configs["positive"], field)]
    audit_rows.append({"forest": pair["label"], "lambda_zero": configs["zero"]["lambda_growth"],
                       "lambda_positive": configs["positive"]["lambda_growth"],
                       "controlled_fields": len(CONTROL_FIELDS), "mismatch_count": len(mismatches),
                       "mismatches": "; ".join(mismatches), "strict_pair_pass": len(mismatches)==0})
audit = pd.DataFrame(audit_rows)
audit.to_csv(TABLES / "01_strict_configuration_audit.csv", index=False)
display(audit)
if not audit.strict_pair_pass.all():
    raise RuntimeError("The lambda comparison is not controlled; inspect mismatches before continuing.")


In [ ]:
WORKFLOW_PATH = ROOT / "Source" / "Project" / "final_phase2_aoi_workflow.py"

def load_site(site):
    name = f"lambda_control_{site}"
    spec = importlib.util.spec_from_file_location(name, WORKFLOW_PATH)
    workflow = importlib.util.module_from_spec(spec); sys.modules[name] = workflow; spec.loader.exec_module(workflow)
    engine = workflow.load_engine(); workflow.configure(engine, site)
    engine.GROWTH_ROOT = (ROOT / "Source" / "Training" / "growthloss_v6").resolve()
    engine.OFFICIAL_REPO = engine.GROWTH_ROOT / "official_source" / "ECHOSAT-main"
    engine.AGADIR_CONFIRMATORY_MODE = False
    candidates = {}
    pair = PAIRS[site]
    for role in ("zero", "positive"):
        cfg = json.loads((run_dir(site, pair[role]) / "config.json").read_text(encoding="utf-8"))
        candidates[pair[role]] = {
            "drop_m": cfg["growth_loss_provenance"]["persistent_drop_m"],
            "K": cfg["growth_loss_provenance"]["persistent_required_consecutive_flags"],
            "lambda_growth": cfg["lambda_growth"], "huber_delta": cfg["huber_delta"],
        }
    if site == "ifran": engine.IFRAN_CANDIDATES = candidates
    else: engine.LOW_CANOPY_CANDIDATES = candidates
    modules = workflow.prepare_modules(engine)
    _, shots, records = engine.build_data(site, modules, include_test=False)
    return engine, modules, shots, records["val"], candidates


In [ ]:
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda:f.read(8*1024*1024), b""): h.update(chunk)
    return h.hexdigest()

def val_predictions(site, role, engine, modules, shots, records):
    candidate = PAIRS[site][role]
    checkpoint = run_dir(site, candidate) / "checkpoints" / "best_compromise.ckpt"
    output = CACHE / f"{site}_{role}_val_unique_nearest.csv.gz"
    lineage = output.with_suffix("").with_suffix(".lineage.json")
    digest = sha256(checkpoint)
    if output.is_file() and lineage.is_file() and json.loads(lineage.read_text())["checkpoint_sha256"] == digest:
        print("[REUSE]", output.name); return pd.read_csv(output)
    cfg = engine.FORESTS[site]; model = engine.fresh_model(cfg, modules)
    try: state=torch.load(checkpoint,map_location="cpu",weights_only=False)
    except TypeError: state=torch.load(checkpoint,map_location="cpu")
    model.prediction_head.load_state_dict(state["prediction_head"], strict=True); model.eval()
    _, nearest = modules["evaluate_full_patch_temporal_nearest"](
        model=model, records=records, shots=shots, device=DEVICE, split="val", drop_channels=(),
        min_height=cfg["eval_min"], max_height=cfg["eval_max"], progress_every=5)
    nearest.to_csv(output,index=False,compression="gzip")
    lineage.write_text(json.dumps({"forest":site,"role":role,"split":"VAL only","checkpoint":str(checkpoint),
                                   "checkpoint_sha256":digest,"test_used":False},indent=2),encoding="utf-8")
    del model,state; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return nearest

def point_metrics(y,p):
    y=np.asarray(y,float); p=np.asarray(p,float); e=p-y
    return {"n":len(y),"mae":np.abs(e).mean(),"rmse":np.sqrt((e*e).mean()),"bias":e.mean(),
            "r2":1-(e*e).sum()/((y-y.mean())**2).sum(),"corr":np.corrcoef(y,p)[0,1],
            "slope":np.polyfit(y,p,1)[0],"std_ratio":np.std(p)/np.std(y)}

accuracy_rows=[]; site_objects={}
if RUN_VAL_INFERENCE:
    for site,pair in PAIRS.items():
        print("\n=====",pair["label"],"=====",flush=True)
        engine,modules,shots,records,candidates=load_site(site); site_objects[site]=(engine,modules,shots,records,candidates)
        frames={role:val_predictions(site,role,engine,modules,shots,records) for role in ("zero","positive")}
        merged=frames["zero"][["aux_shot_uid","rh95","pred_on_growthloss"]].rename(columns={"pred_on_growthloss":"pred_zero"})
        merged=merged.merge(frames["positive"][["aux_shot_uid","rh95","pred_on_growthloss"]].rename(columns={"rh95":"rh95_positive","pred_on_growthloss":"pred_positive"}),on="aux_shot_uid",how="inner",validate="one_to_one")
        if not np.allclose(merged.rh95,merged.rh95_positive): raise RuntimeError(f"{site}: RH95 mismatch")
        merged.to_csv(TABLES/f"02_{site}_paired_val_predictions.csv.gz",index=False,compression="gzip")
        for role in ("zero","positive"):
            accuracy_rows.append({"forest":pair["label"],"role":role,"lambda_temp":0 if role=="zero" else pair["lambda"],**point_metrics(merged.rh95,merged[f"pred_{role}"])})
    accuracy=pd.DataFrame(accuracy_rows); accuracy.to_csv(TABLES/"03_paired_VAL_accuracy_metrics.csv",index=False); display(accuracy)
else:
    print("VAL inference disabled")


In [ ]:
def load_sequence(paths, row_slice, col_slice):
    cubes=[]; original=None
    for path in paths:
        a=np.load(str(path),mmap_mode="r",allow_pickle=False)
        crop=np.array(a[row_slice,col_slice,:],dtype=np.float32,copy=True)
        original=crop.shape[:2] if original is None else original
        t=torch.from_numpy(np.moveaxis(crop,-1,0)); ph=(16-t.shape[-2]%16)%16; pw=(16-t.shape[-1]%16)%16
        if ph or pw: t=F.pad(t,(0,pw,0,ph),mode="replicate")
        cubes.append(t)
    return torch.stack(cubes).unsqueeze(0).to(DEVICE),original

class TemporalAccumulator:
    def __init__(self): self.sum={k:0. for k in ["abs_d1","abs_d2","gt2","gt5","linear_sq"]}; self.n={k:0 for k in self.sum}; self.sequences=0
    def add(self,z):
        # z [T,H,W], already finite and spatially subsampled
        finite=np.isfinite(z).all(axis=0); self.sequences+=1
        d1=np.diff(z,axis=0); m1=np.broadcast_to(finite,d1.shape)
        self.sum["abs_d1"]+=np.abs(d1)[m1].sum(); self.n["abs_d1"]+=m1.sum()
        self.sum["gt2"]+=(np.abs(d1)>2)[m1].sum(); self.n["gt2"]+=m1.sum()
        self.sum["gt5"]+=(np.abs(d1)>5)[m1].sum(); self.n["gt5"]+=m1.sum()
        d2=np.diff(z,n=2,axis=0); m2=np.broadcast_to(finite,d2.shape)
        self.sum["abs_d2"]+=np.abs(d2)[m2].sum(); self.n["abs_d2"]+=m2.sum()
        tt=np.arange(z.shape[0],dtype=float); tc=tt-tt.mean(); slope=(tc[:,None,None]*z).sum(axis=0)/(tc*tc).sum()
        fit=z.mean(axis=0,keepdims=True)+tc[:,None,None]*slope[None]; sq=np.mean((z-fit)**2,axis=0)
        self.sum["linear_sq"]+=sq[finite].sum(); self.n["linear_sq"]+=finite.sum()
    def finish(self):
        return {"mean_abs_first_difference_m":self.sum["abs_d1"]/self.n["abs_d1"],
                "mean_abs_second_difference_m":self.sum["abs_d2"]/self.n["abs_d2"],
                "change_gt_2m_rate":self.sum["gt2"]/self.n["gt2"],"change_gt_5m_rate":self.sum["gt5"]/self.n["gt5"],
                "mean_linear_fit_rmse_m":np.sqrt(self.sum["linear_sq"]/self.n["linear_sq"]),"sequence_occurrences":self.sequences}

@torch.inference_mode()
def temporal_metrics(site,role,engine,modules,records):
    candidate=PAIRS[site][role]; checkpoint=run_dir(site,candidate)/"checkpoints"/"best_compromise.ckpt"
    cfg=engine.FORESTS[site]; model=engine.fresh_model(cfg,modules)
    try: state=torch.load(checkpoint,map_location="cpu",weights_only=False)
    except TypeError: state=torch.load(checkpoint,map_location="cpu")
    model.prediction_head.load_state_dict(state["prediction_head"],strict=True); model.to(DEVICE).eval(); acc=TemporalAccumulator()
    for idx,record in enumerate(records,1):
        first=np.load(str(record.x_paths[0]),mmap_mode="r",allow_pickle=False); H,W=first.shape[:2]; del first
        for r0 in range(0,H,TILE_SIZE):
            for c0 in range(0,W,TILE_SIZE):
                seq,shape=load_sequence(record.x_paths,slice(r0,min(r0+TILE_SIZE,H)),slice(c0,min(c0+TILE_SIZE,W)))
                out=model(seq)[0].float().cpu().numpy(); h,w=shape
                z=out[1,:,:h:SPATIAL_STRIDE,:w:SPATIAL_STRIDE].astype(float); acc.add(z)
                del seq,out
        if idx%5==0 or idx==len(records): print(f"[TEMPORAL VAL] {site} {role} {idx}/{len(records)}",flush=True)
    del model,state; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return acc.finish()

temporal_rows=[]
if RUN_TEMPORAL_DIAGNOSTICS:
    for site,pair in PAIRS.items():
        if site not in site_objects:
            engine,modules,shots,records,candidates=load_site(site)
        else: engine,modules,shots,records,candidates=site_objects[site]
        for role in ("zero","positive"):
            cache_path=CACHE/f"{site}_{role}_temporal_metrics.json"
            digest=sha256(run_dir(site,pair[role])/"checkpoints"/"best_compromise.ckpt")
            if cache_path.is_file():
                payload=json.loads(cache_path.read_text())
                if payload.get("checkpoint_sha256")==digest: result=payload["metrics"]; print("[REUSE]",cache_path.name)
                else: result=temporal_metrics(site,role,engine,modules,records)
            else: result=temporal_metrics(site,role,engine,modules,records)
            cache_path.write_text(json.dumps({"checkpoint_sha256":digest,"metrics":result},indent=2),encoding="utf-8")
            temporal_rows.append({"forest":pair["label"],"role":role,"lambda_temp":0 if role=="zero" else pair["lambda"],**result})
    temporal=pd.DataFrame(temporal_rows); temporal.to_csv(TABLES/"04_paired_VAL_temporal_metrics.csv",index=False); display(temporal)
else: print("Temporal diagnostics disabled")


In [ ]:
if not (TABLES/"03_paired_VAL_accuracy_metrics.csv").is_file() or not (TABLES/"04_paired_VAL_temporal_metrics.csv").is_file():
    raise RuntimeError("Run both VAL accuracy and temporal-diagnostic sections before drawing a conclusion.")
accuracy=pd.read_csv(TABLES/"03_paired_VAL_accuracy_metrics.csv")
temporal=pd.read_csv(TABLES/"04_paired_VAL_temporal_metrics.csv")

def paired_delta(table, metrics):
    rows=[]
    for forest,g in table.groupby("forest"):
        z=g.set_index("role"); row={"forest":forest}
        for m in metrics: row[f"delta_{m}_positive_minus_zero"]=float(z.loc["positive",m]-z.loc["zero",m])
        rows.append(row)
    return pd.DataFrame(rows)

accuracy_delta=paired_delta(accuracy,["mae","rmse","r2","bias","slope","std_ratio"])
temporal_delta=paired_delta(temporal,["mean_abs_first_difference_m","mean_abs_second_difference_m","change_gt_2m_rate","change_gt_5m_rate","mean_linear_fit_rmse_m"])
decision=accuracy_delta.merge(temporal_delta,on="forest")
decision.to_csv(TABLES/"05_lambda_temporal_controlled_deltas.csv",index=False)
display(decision.style.format(precision=5))

fig,axes=plt.subplots(1,2,figsize=(11,4),dpi=150)
for ax,source,metrics,title in [
    (axes[0],accuracy,["mae","rmse"],"GEDI-supported VAL error"),
    (axes[1],temporal,["mean_abs_first_difference_m","mean_abs_second_difference_m"],"Dense temporal diagnostics")]:
    plot=source.melt(id_vars=["forest","role"],value_vars=metrics,var_name="metric",value_name="value")
    labels=[]; values=[]; colors=[]
    for _,r in plot.iterrows(): labels.append(f"{r.forest}\n{r.role}\n{r.metric.replace('mean_abs_','').replace('_m','')}"); values.append(r.value); colors.append("#999999" if r.role=="zero" else "#2c7fb8")
    ax.bar(np.arange(len(values)),values,color=colors); ax.set_xticks(np.arange(len(values))); ax.set_xticklabels(labels,rotation=55,ha="right",fontsize=7); ax.set_title(title); ax.grid(axis="y",alpha=.25)
fig.tight_layout(); fig.savefig(FIGURES/"lambda_temporal_controlled_VAL.png",bbox_inches="tight"); fig.savefig(FIGURES/"lambda_temporal_controlled_VAL.pdf",bbox_inches="tight"); plt.show()

manifest={"split":"VAL only","test_used":False,"comparison":"lambda_temp=0 versus retained positive lambda",
          "strict_configuration_audit":str(TABLES/"01_strict_configuration_audit.csv"),
          "checkpoint_rule":"best_compromise.ckpt for both pair members","spatial_stride_temporal_audit":SPATIAL_STRIDE,
          "interpretation":"accuracy and temporal-coherence comparison; temporal metrics are model-derived diagnostics, not observed annual growth"}
(OUT/"reproducibility_manifest.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")
print("[DONE]",OUT)
